# SCRUM-5: Build Product Registry & Source Log

**Task:** Collect the actual dataset by defining the target products and documenting exactly where each data source came from and how it was collected.

**Assignee:** Burra Vijay Krishna | **Sprint:** SCRUM Sprint 0

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = os.path.join('..', 'data')
print('Setup complete.')

## 1. Product Registry — Loading & Inspection

In [ ]:
products = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))

print(f'Total products in registry: {len(products)}')
print(f'Columns: {list(products.columns)}')
print(f'Unique brands: {products["brand"].nunique()}')
print(f'Category: {products["category"].unique()[0]}')
print()
products.head(10)

In [ ]:
print('--- Product Type Distribution ---')
print(products['product_type'].value_counts())
print()
print('--- Brand Distribution ---')
print(products['brand'].value_counts())

## 2. Product Type & Brand Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = products['product_type'].value_counts()
colors_type = {'RISING': '#e74c3c', 'STABLE': '#2ecc71', 'DECLINING': '#3498db'}
axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.0f%%',
            colors=[colors_type.get(t, '#95a5a6') for t in type_counts.index],
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Product Type Distribution', fontsize=14, fontweight='bold')

brand_counts = products['brand'].value_counts()
bars = axes[1].barh(brand_counts.index[::-1], brand_counts.values[::-1],
                    color=sns.color_palette('muted', len(brand_counts)))
axes[1].set_xlabel('Number of Products')
axes[1].set_title('Products per Brand', fontsize=14, fontweight='bold')
for bar in bars:
    w = bar.get_width()
    axes[1].text(w + 0.2, bar.get_y() + bar.get_height()/2, f'{int(w)}',
                ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Brand x Product Type Heatmap

In [ ]:
brand_type = pd.crosstab(products['brand'], products['product_type'])
for col in ['RISING', 'STABLE', 'DECLINING']:
    if col not in brand_type.columns:
        brand_type[col] = 0
brand_type = brand_type[['RISING', 'STABLE', 'DECLINING']]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(brand_type, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Number of Products'}, ax=ax)
ax.set_title('Brand x Product Type Matrix', fontsize=14, fontweight='bold')
ax.set_ylabel('Brand')
ax.set_xlabel('Product Type')
plt.tight_layout()
plt.show()

## 4. Launch Timeline

In [ ]:
launch_data = products[~products['launch_period'].isin(['established', 'unverified'])].copy()
launch_counts = launch_data['launch_period'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(launch_counts.index, launch_counts.values,
              color=sns.color_palette('viridis', len(launch_counts)))
ax.set_xlabel('Launch Month')
ax.set_ylabel('Number of Products')
ax.set_title('Product Launch Timeline (India Market)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.xticks(rotation=45)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{int(h)}',
            ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Data Sources Documentation

In [ ]:
source_log = pd.read_csv(os.path.join(DATA_DIR, 'source_log.csv'))
print(f'Data sources documented: {len(source_log)}')
print()
for i, row in source_log.iterrows():
    print(f'Source {i+1}: {row["source"]}')
    print(f'  Access: {str(row["access_method"])[:100]}...')
    print(f'  Granularity: {row["granularity"]}')
    print(f'  Added: {row["added_date"]}')
    print()

## 6. Collection Log Summary

In [ ]:
collection_log = pd.read_csv(os.path.join(DATA_DIR, 'collection_log.csv'))
print(f'Total collection runs: {len(collection_log)}')
print(f'All successful: {(collection_log["status"].isin(["success", "smoke_test_ok"])).all()}')
print()

summary = collection_log.groupby('source').agg(
    runs=('status', 'count'),
    total_records=('records_collected', 'sum'),
    total_quota=('quota_units', 'sum')
).reset_index()
print(summary.to_string(index=False))
print()

collected = collection_log[collection_log['status'] == 'success']['product_id'].unique()
print(f'Pilot products with YouTube data: {len(collected)}')
print(f'IDs: {sorted(collected)}')

In [ ]:
print('\n=== SCRUM-5 COMPLETE ===')
print('Product registry: 60 smartphones documented')
print('Data sources: YouTube Data API v3 + Google Trends (manual export)')
print('Collection: 10 pilot products collected successfully')